# Model predictions

Applies the trained classifier to every site-month, writes per-pixel class rasters, and
aggregates them into the per-waterhole composition time series the project exists to produce.

**Outputs**

| file | what it is |
|---|---|
| `predictions/pixel_predictions/site_XXX/*_pred.tif` | uint8 class raster on the source tile's exact grid |
| `..._conf.tif` | prediction confidence, 0-100 |
| `..._pred.png` | the classes colourised, for display |
| `..._rgb.png` | true colour — what the classifier actually saw |
| `..._conf.png` | confidence colourised — how much to believe it |
| `..._/bounds.json` | WGS84 bounds and the layer list, written once per site |
| `predictions/waterhole_composition.csv` | one row per site-month: counts and fractions per class |
| `predictions/waterhole_boxes.geojson` | all 187 boxes, for the dashboard map |
| `predictions/class_colours.json` | the class scheme and confidence ramp, so a legend cannot drift |

The PNGs and `bounds.json` exist so a browser can place predictions on a map as image
overlays with no tile server and no backend — see `dashboard_plan.md`.

All three PNG layers share one grid and one `bounds.json`, so the dashboard can flip between
them without moving anything. That is the point of the RGB and confidence layers: a class
overlay on a satellite basemap is unfalsifiable on its own, because the basemap is a
different sensor from a different year. Showing what the classifier saw, and how sure it
was, is what lets a viewer check a prediction rather than take it on trust.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "cookie-cutting" else Path.cwd() / "cookie-cutting"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wh_bbox
import wh_config
import wh_features
import wh_inventory
import wh_plots
import wh_predict
import wh_train

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

cfg = wh_config.load()
manifest = wh_inventory.load_manifest(cfg)
print("config", cfg.source_path.name, "hash", cfg.hash)
print(f"{len(manifest):,} chips, {manifest['site_id'].nunique()} sites, "
      f"{manifest['year_month'].nunique()} months")

## Parameters

In [ ]:
PARAMS = wh_predict.PredictParams(
    model_name="classifier",          # from derived/models/

    # 3x3 majority filter, OFF. Standard despeckling for landscape classification
    # and wrong at this scale: 39% of the water patches here are 1-2 px and the
    # filter deletes them. Set to 3 if a site looks noisy and you want it.
    majority_filter_px=0,

    # Confidence band: max class probability as uint8 0-100. Costs roughly 60% of
    # the run time, because predict_proba is the expensive call. It is also the
    # only signal for which of the 160 sites the model never saw are worth
    # distrusting, and the "conf" display layer cannot be written without it.
    write_confidence=True,

    # Display images written beside each GeoTIFF, all sharing one grid and one
    # bounds.json:
    #   pred  the classification, class 0 transparent
    #   rgb   true colour, what the classifier actually saw
    #   conf  max class probability, how sure it was
    png_layers=("pred", "rgb", "conf"),

    # "png" or "webp". Measured per site-month on a 150x151 chip:
    #
    #            PNG      WebP
    #   pred    2.1 KB    0.9 KB
    #   conf   18.6 KB    4.8 KB
    #   rgb    49.4 KB    4.7 KB   <- true colour is photographic and compresses
    #                                 terribly as PNG
    #
    # Over 15,708 site-months that is ~1.2 GB against ~150 MB, which is the
    # difference between a static site that can be hosted anywhere and one that
    # cannot. Only the rgb layer is encoded lossily; class and confidence images
    # stay lossless in either format, so no pixel ever decodes to a wrong class.
    # PNG is the default only because it is what is already on disk.
    image_format="png",

    # Which region the plots below summarise. BOTH bbox and footprint counts are
    # always written to the CSV, so switching this needs no re-run.
    denominator="bbox",

    # Flags an isolated wet month sitting between dry ones on a thin median.
    # It flags and never rewrites: it might be a compositing artefact, or it
    # might be the rainfall event you are looking for.
    flag_isolated_wet=True,
    low_obs_threshold=2.0,

    sites=None,        # None = all 187, or a tuple like ("025", "002")
    overwrite=False,   # False leaves files that already exist alone
    workers=6,         # sites are independent; 6 workers ~ 70 min for the archive
)

PARAMS

## Check what the model actually is

The manifest is the authority on how this model must be applied. A model given features in a
different order does not fail — it returns confident nonsense — so the feature definition is
rebuilt from the manifest rather than from anything in this notebook.

In [ ]:
model, meta = wh_train.load_model(cfg, PARAMS.model_name)
wh_predict.check_scheme(meta, cfg)

print(f"model      : {meta['model_class']}")
print(f"trained    : {meta['timestamp']}")
print(f"features   : {meta['n_features']}")
print(f"CV         : {meta['cv_macro_f1']:.3f} macro F1 ({meta['cv_strategy']})")
print(f"labels     : {meta['n_training_pixels']:,} px over {len(meta['training_sites'])} sites")

feature_params = wh_predict.feature_params_from_manifest(meta)
print(f"\nembeddings : {feature_params.use_alphaearth} "
      f"bands={feature_params.alphaearth_bands or 'all 64'}")
print(f"n_obs      : {feature_params.include_n_obs}")

**The model was trained on a subset of the sites and is about to be applied to all of
them.** Waterholes unlike anything labelled will still receive a confident-looking answer.
`mean_confidence` in the CSV, and the `_conf.tif` rasters, are what make that visible —
check them before trusting a site you never labelled.

## Dry run on one site

Predict a single site and look at it before committing an hour to the archive.

In [ ]:
DRY_RUN_SITE = "025"

boxes = wh_bbox.load_boxes(cfg)
records = wh_predict.predict_site(
    manifest, cfg, model, meta, PARAMS, DRY_RUN_SITE, boxes
)
preview = pd.DataFrame(records)
print(f"{len(preview)} months predicted for site {DRY_RUN_SITE}")
preview[["year_month", "bbox_n_classified", "n_pixels_footprint",
         "mean_confidence", "gap_fraction"]].head()

In [ ]:
preview

In [ ]:
import rasterio

import wh_footprint
import wh_tiles

# A mid-confidence month, alongside its true colour and confidence. These three
# panels are exactly what the dashboard shows per month, from the same files.
sample = preview.sort_values("mean_confidence").iloc[len(preview) // 2]["year_month"]
row = manifest[(manifest["site_id"] == DRY_RUN_SITE)
               & (manifest["year_month"] == sample)].iloc[0]
tile = wh_tiles.read_tile(row["tif_path"], cfg)
stem = Path(row["tif_path"]).stem

with rasterio.open(wh_predict.raster_path(cfg, DRY_RUN_SITE, stem, "pred")) as src:
    classes = src.read(1)
with rasterio.open(wh_predict.raster_path(cfg, DRY_RUN_SITE, stem, "conf")) as src:
    confidence = src.read(1) / 100.0

# The two regions the composition is counted within.
box_mask = wh_bbox.load_mask(cfg, DRY_RUN_SITE)
try:
    footprint_mask = wh_footprint.load_mask(cfg, DRY_RUN_SITE)
except FileNotFoundError:
    footprint_mask = None
    print(f"site {DRY_RUN_SITE} has no footprint; only the bounding box is drawn")

figure, axes = plt.subplots(1, 3, figsize=(13, 4.4), constrained_layout=True)

axes[0].imshow(wh_plots.rgb_composite(tile))
axes[0].set_title(f"RGB {sample}", fontsize=9)

axes[1].imshow(wh_plots.rgb_composite(tile))
axes[1].imshow(wh_plots.class_overlay(classes, cfg, alpha=0.8), interpolation="nearest")
axes[1].set_title("predicted", fontsize=9)

image = axes[2].imshow(confidence, cmap=wh_predict.CONFIDENCE_CMAP,
                       vmin=wh_predict.CONFIDENCE_VMIN, vmax=wh_predict.CONFIDENCE_VMAX)
plt.colorbar(image, ax=axes[2], fraction=0.046, pad=0.03)
axes[2].set_title(f"confidence (median {np.median(confidence):.2f})", fontsize=9)

# Yellow dotted = the labelled bounding box, cyan = the derived footprint.
# The bbox_* columns count within the yellow outline, footprint_* within the cyan.
for axis in axes:
    if box_mask.any():
        axis.contour(box_mask, levels=[0.5], colors="#ffff00",
                     linewidths=1.0, linestyles="dotted")
    if footprint_mask is not None and footprint_mask.any():
        axis.contour(footprint_mask, levels=[0.5], colors="#00ffff", linewidths=1.2)
    axis.set_xticks([]); axis.set_yticks([])

wh_plots.class_legend(axes[-1], cfg)
figure.suptitle(
    f"site {DRY_RUN_SITE} {sample}   "
    f"yellow dotted = bounding box, cyan = footprint", fontsize=10,
)
plt.show()

### The images the dashboard will actually load

The figure above is drawn from arrays in memory. This one reads the three images back off
disk and places them on their `bounds.json`, which is what the browser does — so if the
colours, the transparency or the alignment are wrong, they are wrong here too.

The outlines are drawn *over* the images, not baked into them, which is also how the
dashboard should do it: a site's box and footprint are identical for all 84 of its months, so
burning them into 15,708 images would repeat them needlessly, destroy the pixels underneath,
and leave them impossible to switch off. They ship as two GeoJSON map layers instead — see
the export cell at the end.

In [ ]:
import json as _json

from PIL import Image

bounds = _json.loads((wh_predict.site_dir(cfg, DRY_RUN_SITE) / "bounds.json").read_text())
layers = bounds["png_layers"]
print(f"layers : {layers}  ({bounds['image_format']})")
print(f"bounds : {bounds['leaflet_bounds']}")

figure, axes = plt.subplots(1, len(layers), figsize=(4.3 * len(layers), 4.4),
                            constrained_layout=True)
axes = np.atleast_1d(axes)

for axis, layer in zip(axes, layers):
    path = wh_predict.raster_path(
        cfg, DRY_RUN_SITE, stem, f"{layer}_png", bounds["image_format"]
    )
    if not path.exists():
        axis.text(0.5, 0.5, f"{layer}\nnot written", ha="center", va="center",
                  fontsize=9, color="#666666", transform=axis.transAxes)
        axis.set_xticks([]); axis.set_yticks([])
        continue

    # Chequerboard behind, so transparent pixels read as transparent rather
    # than as white.
    board = np.indices((16, 16)).sum(axis=0) % 2
    axis.imshow(board, cmap="binary", vmin=-2, vmax=3,
                extent=(0, tile.shape[1], tile.shape[0], 0), interpolation="nearest")
    axis.imshow(np.array(Image.open(path).convert("RGBA")), interpolation="nearest")
    axis.set_title(f"{path.name.split('_')[-1]}  "
                   f"({path.stat().st_size / 1024:.0f} KB)", fontsize=9)

    # The two counting regions, same colours as the figure above: yellow dotted
    # = bounding box (the bbox_* columns), cyan = footprint (footprint_*).
    # box_mask and footprint_mask come from the previous cell.
    if box_mask.any():
        axis.contour(box_mask, levels=[0.5], colors="#ffff00",
                     linewidths=1.0, linestyles="dotted")
    if footprint_mask is not None and footprint_mask.any():
        axis.contour(footprint_mask, levels=[0.5], colors="#00ffff", linewidths=1.2)
    axis.set_xticks([]); axis.set_yticks([])

figure.suptitle(
    f"site {DRY_RUN_SITE} {sample} — as served to the browser   "
    f"(yellow dotted = bounding box, cyan = footprint)", fontsize=10,
)
plt.show()

## Run the archive

**This is the long one — 30 to 70 minutes depending on `write_confidence`.** It reports as it
goes rather than going silent:

- how many sites still need files written, before any work starts
- a bar that advances as each site *finishes*, with elapsed time, a running ETA and the
  cumulative site-month count
- any failure the moment it happens, named, without stopping the rest

The first sites are slower than the rest: every worker loads the config, manifest and model
once before its first site. A slow start is not a hang.

**Restartable rather than resumable.** Every site is predicted again on a re-run, but a file
that already exists is not rewritten unless `overwrite=True`. So an interrupted run finishes
correctly, and adding a display layer later writes only that layer — but the prediction
itself, which is where the time goes, happens either way. Every site has to be visited
regardless, because the composition record is built from the prediction rather than read back
off disk. If only a display layer is missing, use the backfill cell below instead: it needs
no model and takes about two minutes.

In [ ]:
table = wh_predict.run(manifest, cfg, PARAMS)
path = wh_predict.save_table(table, cfg)

print(f"\n{len(table):,} site-months -> {path}")
print(f"{table['site_id'].nunique()} sites, {table['year_month'].nunique()} months")
table.head(3)

### Adding a display layer to an archive already predicted

If the class rasters exist and only a display layer is missing, there is no need to predict
anything again: `rgb` comes from the source chip and `pred`/`conf` from the rasters on disk.
About **2 minutes** for all 187 sites, against ~70 for a full re-run. It also works for
re-encoding — switching `image_format` and running this writes the new files without touching
the model.

The one thing it cannot do is invent confidence. `_conf.tif` is written only when
`write_confidence=True`, and it comes from `predict_proba` — so if the archive was run
without it, the confidence overlay needs a real re-run of the cell above. The cell reports
exactly how many months are in that position.

In [ ]:
# Safe to run at any time: it only writes files that are missing.
totals = wh_predict.backfill_pngs(manifest, cfg, PARAMS)

## Sanity checks

Worth running before drawing any conclusions from the table.

In [ ]:
names = [d.name for d in cfg.classes if not d.ignore]
fractions = table[[f"bbox_frac_{n}" for n in names]].sum(axis=1, min_count=1)
classified = table["bbox_n_classified"] > 0

print(f"rows                          : {len(table):,}")
print(f"fractions sum to 1 where any  : "
      f"{bool(((fractions - 1).abs() < 1e-9)[classified].all())}")
print(f"site-months with no classified px: {int((~classified).sum())}")
print(f"sites with no footprint       : "
      f"{int((~table.groupby('site_id')['has_footprint'].any()).sum())}")
print(f"counts never exceed the region: "
      f"{bool((table['bbox_n_classified'] <= table['n_pixels_bbox']).all())}")
print(f"\ndata quality: {table['data_quality'].value_counts().to_dict()}")
print(f"isolated-wet flags: {int(table['flag_isolated_wet'].sum())}")

## Composition through time

The stacked area for one waterhole. Flagged months are marked with a star and months with no
classified pixels with a grey triangle — neither is removed, because a gap in the record is
information too.

In [ ]:
figure = wh_plots.plot_site_composition(table, DRY_RUN_SITE, cfg, PARAMS.denominator)
plt.show()

In [ ]:
# A few more, chosen to span the waterhole types in the box labels.
for site_id in table.drop_duplicates("site_id").head(4)["site_id"]:
    if site_id == DRY_RUN_SITE:
        continue
    figure = wh_plots.plot_site_composition(table, site_id, cfg, PARAMS.denominator)
    plt.show()

### Every site at once

Seasonality should read as vertical banding. A site that looks unlike its neighbours is
either genuinely different or one the model does not handle — the confidence panel below is
how to tell those apart.

In [ ]:
figure = wh_plots.plot_composition_heatmap(
    table, cfg, class_name="aquatic_vegetation", denominator=PARAMS.denominator
)
plt.show()

In [ ]:
figure = wh_plots.plot_composition_quality(table)
plt.show()

by_site = table.groupby("site_id")["mean_confidence"].mean().dropna()
if by_site.empty:
    print("No confidence was recorded — write_confidence is False in PredictParams.")
    print("That halves the run time but removes the only signal for which sites the")
    print("model handles badly, and it is applied to far more sites than it was trained on.")
else:
    print("lowest-confidence sites — check these before trusting their series:")
    print(by_site.nsmallest(8).round(3).to_string())

## Export the dashboard assets

Four static files: the box polygons, the footprint polygons, the class scheme and the
confidence ramp — everything `dashboard_plan.md` expects beside the per-site images.

In [ ]:
geojson_path = wh_predict.export_boxes_geojson(cfg, boxes)
footprint_path = wh_predict.export_footprints_geojson(cfg, sorted(boxes.index))
colours_path = wh_predict.export_class_colours(cfg)

print(f"boxes      -> {geojson_path.name}")
print(f"footprints -> {footprint_path.name}")
print(f"colours    -> {colours_path.name}   (class colours + the confidence ramp)")

footprints = _json.loads(footprint_path.read_text())
absent = footprints["sites_without_footprint"]
print(f"\n{len(footprints['features'])} footprints; {len(absent)} sites have none: "
      f"{', '.join(absent) if absent else '-'}")
print("Those sites still have a bounding box, so bbox_* is defined everywhere and")
print("footprint_* is not — which is why both denominators are in the CSV.")

root = cfg.paths["predictions"] / "pixel_predictions"
sizes: dict[str, tuple[int, float]] = {}
for path in root.rglob("*"):
    if path.is_file():
        kind = path.name.split("_")[-1]
        count, total = sizes.get(kind, (0, 0.0))
        sizes[kind] = (count + 1, total + path.stat().st_size)

print("\npixel_predictions on disk:")
for kind, (count, total) in sorted(sizes.items()):
    print(f"  {kind:<12} {count:>6,} files  {total / 1e6:>7.1f} MB")
print(f"  {'total':<12} {sum(c for c, _ in sizes.values()):>6,} files  "
      f"{sum(t for _, t in sizes.values()) / 1e6:>7.1f} MB")
print("\nSee dashboard_plan.md — image_format='webp' cuts this roughly 5x if it is")
print("larger than you want to host.")